# FE용 데이터셋 제작
1. 고장 개체 비율 fs_train 8 : fs_validation 2 (홀드아웃)
2. serial_number 단위에서 failure 비율 유지하여 분할
3. 학습 및 테스트 세트 모두 정상 개체는 고장 개체수의 1배수(1:1) 샘플링하여 배정
4. 
    - 개체 단위 1:1 비율 적용
- (같은 serial_number는 train과 validation에 동시에 존재하면 안 됨)
- seed = 42

In [1]:
import duckdb
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# ==========================================
# ⚙️ 실행 설정 (원하는 작업만 True로 바꾸세요)
# ==========================================
CONFIG = {
    "BUILD_TRAIN": True,  # 트레인 세트 제작 여부
    "BUILD_VAL": True,    # 테스트 세트 제작 여부
    "TRAIN_RATIO": 1,     # 트레인 정상 샘플링 배수 (1:1)
    "VAL_RATIO": 1,      # 테스트 정상 샘플링 배수 (1:1)
}

# 1. 경로 설정
base_dir = r'../data2/04_feature_engineering'
target_file_name = 'fs_diff.parquet'
target_path = os.path.join(base_dir, target_file_name)
output_train = os.path.join(base_dir, 'fs_train.parquet')
output_val = os.path.join(base_dir, 'fs_validation.parquet')

# 2. 파일 필터링
all_files = [f for f in os.listdir(base_dir) if f.endswith('.parquet')]
feature_files = [
    f for f in all_files 
    if not f.startswith('tmp_') 
    and 'train' not in f.lower() 
    and 'validation' not in f.lower() 
    and f != target_file_name
]

con = duckdb.connect()
# [메모리 가드] OOM 방지를 위한 안전 설정
con.execute("SET memory_limit = '20GB'")
con.execute("SET threads = 4")

# [Step 0] 청소 및 메타데이터 캐싱
print("🧹 충돌 방지를 위한 파일 청소 및 메타데이터 캐싱 중...")
for f_name, build_flag in [('fs_train.parquet', CONFIG["BUILD_TRAIN"]), ('fs_validation.parquet', CONFIG["BUILD_VAL"])]:
    if build_flag and f_name in os.listdir(base_dir):
        try: os.remove(os.path.join(base_dir, f_name))
        except: pass

file_meta = {}
for f in [target_file_name] + feature_files:
    f_path = os.path.join(base_dir, f)
    v_name = f"v_{f.replace('.', '_').replace('-', '_')}"
    con.execute(f"CREATE OR REPLACE VIEW {v_name} AS SELECT * FROM read_parquet('{f_path}')")
    cols = con.execute(f"SELECT * FROM {v_name} WHERE 1=0").df().columns.tolist()
    file_meta[f] = {"view": v_name, "columns": cols}

# [Step 1] SN 개체 단위 8:2 층화 분할 및 정상 개체 10배수 샘플링 (seed=42)
target_view = file_meta[target_file_name]["view"]

# 모든 고장 개체와 정상 개체 목록 분리 (MAX(failure) 기준)
serial_stats = con.execute(f"SELECT serial_number, MAX(failure) as has_failed FROM {target_view} GROUP BY serial_number").df()

failed_serials = serial_stats[serial_stats['has_failed'] == 1]['serial_number'].reset_index(drop=True)
healthy_serials = serial_stats[serial_stats['has_failed'] == 0]['serial_number'].reset_index(drop=True)

# 1. 고장 개체 8:2 홀드아웃 분할 (random_state=42)
train_failed, val_failed = train_test_split(failed_serials, test_size=0.2, random_state=42)

# 2. 정상 개체 8:2 분할 (train/validation 간 개체 누수 방지, random_state=42)
train_healthy_pool, val_healthy_pool = train_test_split(healthy_serials, test_size=0.2, random_state=42)

# 3. 각 세트별로 정상 개체를 고장 개체의 10배수 샘플링 (random_state=42)
sampled_train_healthy = train_healthy_pool.sample(n=len(train_failed) * CONFIG["TRAIN_RATIO"], random_state=42)
sampled_val_healthy = val_healthy_pool.sample(n=len(val_failed) * CONFIG["VAL_RATIO"], random_state=42)

# 4. 고장 개체와 샘플링된 정상 개체 병합
train_sn = pd.concat([train_failed, sampled_train_healthy]).reset_index(drop=True)
val_sn = pd.concat([val_failed, sampled_val_healthy]).reset_index(drop=True)

# 5. DuckDB에 임시 등록
con.register('train_sn_list', pd.DataFrame({'serial_number': train_sn}))
con.register('val_sn_list', pd.DataFrame({'serial_number': val_sn}))

def build_query(sn_table, output_file):
    """지정된 개체 리스트에 대해 전체 피처 병합하여 쿼리 빌드"""
    target_cols = file_meta[target_file_name]["columns"]
    seen_columns = set(target_cols)
    
    # 메인 타겟 테이블 필터링 뷰 생성
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW tmp_target_filtered AS 
        SELECT * FROM {target_view} 
        WHERE serial_number IN (SELECT serial_number FROM {sn_table})
    """)
    
    select_parts = ["s.*"]
    join_parts = []
    
    for i, f in enumerate(feature_files):
        v_name = file_meta[f]["view"]
        
        # 각 피처 파일 필터링 뷰 생성
        tmp_view_name = f"tmp_feat_filtered_{i}"
        con.execute(f"""
            CREATE OR REPLACE TEMP VIEW {tmp_view_name} AS 
            SELECT * FROM {v_name} 
            WHERE serial_number IN (SELECT serial_number FROM {sn_table})
        """)
        
        current_cols = file_meta[f]["columns"]
        to_exclude = [col for col in current_cols if col in seen_columns]
        exclude_clause = ", ".join([f'"{c}"' for c in to_exclude])
        
        if to_exclude:
            select_parts.append(f'{tmp_view_name}.* EXCLUDE ({exclude_clause})')
        else:
            select_parts.append(f"{tmp_view_name}.*")
            
        join_parts.append(f"LEFT JOIN {tmp_view_name} USING (serial_number, date)")
        seen_columns.update(current_cols)
        
    main_sql = f"""
    SELECT {", ".join(select_parts)}
    FROM tmp_target_filtered s
    {"".join(join_parts)}
    """
    return f"COPY ({main_sql}) TO '{output_file}' (FORMAT 'PARQUET')"

# [Step 2] 실행 (CONFIG 설정에 따라 분기)
if CONFIG["BUILD_TRAIN"]:
    print(f"🏗️ [Train] 제작 시작 (개체 단위 정상 샘플링 1:{CONFIG['TRAIN_RATIO']})")
    con.execute(build_query("train_sn_list", output_train))

if CONFIG["BUILD_VAL"]:
    print(f"🏗️ [Validation] 제작 시작 (개체 단위 정상 샘플링 1:{CONFIG['VAL_RATIO']})")
    con.execute(build_query("val_sn_list", output_val))

# [검증]
print("\n⛪ [검증] 최종 데이터 분포 확인")
for name, path, run in [("TRAIN", output_train, CONFIG["BUILD_TRAIN"]), ("VAL", output_val, CONFIG["BUILD_VAL"])]:
    if run:
        stats = con.execute(f"SELECT SUM(CASE WHEN failure=1 THEN 1 ELSE 0 END) as fail, SUM(CASE WHEN failure=0 THEN 1 ELSE 0 END) as healthy FROM read_parquet('{path}')").df()
        print(f"[{name} 행수] 고장: {stats['fail'][0]:,} / 정상: {stats['healthy'][0]:,} (행 비율 1:{round(stats['healthy'][0]/stats['fail'][0], 1)})")
        
        # 개체 수 검증 추가
        drives_stats = con.execute(f"SELECT COUNT(DISTINCT CASE WHEN failure=1 THEN serial_number END) as fail_drives, COUNT(DISTINCT CASE WHEN failure=0 THEN serial_number END) as healthy_drives FROM read_parquet('{path}')").df()
        print(f"[{name} 개체수] 고장: {drives_stats['fail_drives'][0]:,} / 정상: {drives_stats['healthy_drives'][0]:,} (개체 비율 1:{round(drives_stats['healthy_drives'][0]/drives_stats['fail_drives'][0], 1)})")

con.close()
print("\n🚀 설정된 작업이 모두 완료되었습니다!")


🧹 충돌 방지를 위한 파일 청소 및 메타데이터 캐싱 중...
🏗️ [Train] 제작 시작 (개체 단위 정상 샘플링 1:1)
🏗️ [Validation] 제작 시작 (개체 단위 정상 샘플링 1:1)

⛪ [검증] 최종 데이터 분포 확인
[TRAIN 행수] 고장: 82,824.0 / 정상: 5,075,914.0 (행 비율 1:61.3)
[TRAIN 개체수] 고장: 2,824 / 정상: 5,493 (개체 비율 1:1.9)
[VAL 행수] 고장: 20,993.0 / 정상: 1,230,912.0 (행 비율 1:58.6)
[VAL 개체수] 고장: 707 / 정상: 1,379 (개체 비율 1:2.0)

🚀 설정된 작업이 모두 완료되었습니다!


In [4]:
import duckdb
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# 1. 필수 경로 재설정
base_dir = r'../data2/04_feature_engineering'
target_file_name = 'fs_diff.parquet'
target_path = os.path.join(base_dir, target_file_name)

# 2. DuckDB 연결
con = duckdb.connect()

print("🔍 데이터 로드 및 8:2 분할 시뮬레이션 중...")

# 3. 전체 시리얼 번호(SN) 목록과 고장 여부 가져오기
serial_stats = con.execute(f"""
    SELECT serial_number, MAX(failure) as has_failed 
    FROM read_parquet('{target_path}') 
    GROUP BY serial_number
""").df()

# 4. 8:2로 분할하여 '테스트 그룹' SN만 추출 (random_state 고정)
_, val_sn = train_test_split(
    serial_stats['serial_number'], 
    test_size=0.2, 
    stratify=serial_stats['has_failed'], 
    random_state=42
)

# 5. 테스트 SN 목록을 DuckDB에 임시 등록
con.register('val_sn_temp', pd.DataFrame({'serial_number': val_sn}))

# 6. 해당 그룹의 모든 행(전수조사)에 대한 불균형 비율 계산
result = con.execute(f"""
    SELECT 
        COUNT(CASE WHEN failure = 1 THEN 1 END) as fail_rows,
        COUNT(CASE WHEN failure = 0 THEN 1 END) as healthy_rows,
        ROUND(COUNT(CASE WHEN failure = 0 THEN 1 END) / NULLIF(COUNT(CASE WHEN failure = 1 THEN 1 END), 0), 2) as imbalance_ratio
    FROM read_parquet('{target_path}')
    WHERE serial_number IN (SELECT serial_number FROM val_sn_temp)
""").df()

print("\n📊 [테스트 SN 그룹(전수조사) 불균형 결과]")
print("-" * 40)
print(result.to_string(index=False))
print("-" * 40)
print(f"💡 결론: 테스트 개체들을 전수 조사하면 1:{result['imbalance_ratio'][0]} 비율이 나옵니다.")

con.close()


🔍 데이터 로드 및 8:2 분할 시뮬레이션 중...

📊 [테스트 SN 그룹(전수조사) 불균형 결과]
----------------------------------------
 fail_rows  healthy_rows  imbalance_ratio
     20998       9369766           446.22
----------------------------------------
💡 결론: 테스트 개체들을 전수 조사하면 1:446.22 비율이 나옵니다.


## 3. RFE 샘플링 무결성 검증 테스트 (Verification Validations)

생성된 훈련용(Train) 및 테스트용(Validation) 샘플링 데이터셋의 파일 존재 여부, 두 데이터셋 간 개체(serial_number) 중복 누수 차단 여부, 그리고 고장 개체 대비 정상 개체의 10배수 샘플링 비율을 수학적으로 엄밀히 입증합니다.

In [5]:
import duckdb
import os

con = duckdb.connect()

output_train = r"../data2/04_feature_engineering/fs_train.parquet"
output_val = r"../data2/04_feature_engineering/fs_validation.parquet"

print("🔍 [5-A단계 무결성 검증] 시작...")

try:
    # 1. 파일 존재 확인 검증
    print("Validation 1: 출력 파일 존재 확인 검증")
    assert os.path.exists(output_train), f"오류: {output_train} 파일이 생성되지 않았습니다"
    assert os.path.exists(output_val), f"오류: {output_val} 파일이 생성되지 않았습니다"
    print("  -> [PASS] 출력 파일 존재 확인.")

    # 2. Train / Validation 간 개체 중복 침수(Leakage) 차단 검증
    print("Validation 2: Train과 Validation 간 개체(serial_number) 중복 침수 검증")
    overlap_count = con.execute(f"""
        SELECT COUNT(DISTINCT tr.serial_number)
        FROM read_parquet('{output_train}') tr
        JOIN read_parquet('{output_val}') te ON tr.serial_number = te.serial_number
    """).fetchone()[0]
    assert overlap_count == 0, f"오류: Train과 Validation 셋 사이에 중복되는 serial_number가 {overlap_count}개 존재합니다"
    print("  -> [PASS] Train/Validation 간 개체 침수 없음.")

    # 3. 개체 비율 1:1 검증
    print("Validation 3: 고장 개체와 정상 개체의 1배수(1:1) 샘플링 비율 검증")
    for name, path in [("TRAIN", output_train), ("VAL", output_val)]:
        drives_stats = con.execute(f"""
            WITH MaxFailurePerDrive AS (
                SELECT serial_number, MAX(failure) as max_fail
                FROM read_parquet('{path}')
                GROUP BY serial_number
            )
            SELECT 
                COUNT(CASE WHEN max_fail = 1 THEN 1 END) as fail_drives,
                COUNT(CASE WHEN max_fail = 0 THEN 1 END) as healthy_drives
            FROM MaxFailurePerDrive
        """).fetchone()
        fail_drives, healthy_drives = drives_stats[0], drives_stats[1]
        
        expected_healthy = fail_drives * 1
        assert healthy_drives == expected_healthy, f"오류: {name} 셋의 정상 개체({healthy_drives})가 고장 개체({fail_drives})의 1배({expected_healthy})가 아닙니다!"
    print("  -> [PASS] 고장 개체 대비 정상 개체 1배수 샘플링 통합성 확인.")

    # 4. 결측치 제로 검증
    print("Validation 4: 필수 컬럼 결측치 존재 여부 검증")
    for name, path in [("TRAIN", output_train), ("VAL", output_val)]:
        null_counts = con.execute(f"""
            SELECT 
                COUNT(*) - COUNT(serial_number) as null_sn,
                COUNT(*) - COUNT(date) as null_dt,
                COUNT(*) - COUNT(failure) as null_fl
            FROM read_parquet('{path}')
        """).fetchone()
        assert sum(null_counts) == 0, f"오류: {name} 셋에 결측치가 존재합니다: {null_counts}"
    print("  -> [PASS] 필수 메타 컬럼 결측치 없음.")

    # ── 강화 5: row 수 > 0 검증 ──
    print("Validation 5: 각 파일 row 수 > 0 검증")
    for name, path in [("TRAIN", output_train), ("VAL", output_val)]:
        cnt = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
        assert cnt > 0, f"오류: {name} 파일이 비어 있습니다"
        print(f"  -> {name}: {cnt:,} rows")
    print("  -> [PASS] 모든 파일에 데이터 존재.")

    # ── 강화 6: 스키마 일치 검증 ──
    print("Validation 6: Train/Validation 스키마 일치 검증")
    train_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{output_train}')").fetchdf()['column_name'].tolist()
    validation_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{output_val}')").fetchdf()['column_name'].tolist()
    assert train_cols == validation_cols, f"오류: Train/Validation 스키마 불일치!\n  Train: {train_cols}\n  Validation: {validation_cols}"
    print("  -> [PASS] Train/Validation 스키마 완전 일치.")

    # ── 강화 7: NaN/Inf 검증 ──
    print("Validation 7: NaN/Inf 잔존 검증")
    for name, path in [("TRAIN", output_train), ("VAL", output_val)]:
        numeric_cols = [c for c in train_cols if c not in ('serial_number', 'date')]
        nan_checks = [f"SUM(CASE WHEN isnan({nc}) OR isinf({nc}) THEN 1 ELSE 0 END)" for nc in numeric_cols]
        if nan_checks:
            nan_results = con.execute(f"SELECT {', '.join(nan_checks)} FROM read_parquet('{path}')").fetchone()
            bad_cols = [numeric_cols[j] for j, v in enumerate(nan_results) if v and v > 0]
            assert len(bad_cols) == 0, f"오류: {name}에 NaN/Inf 잔존 컬럼: {bad_cols}"
    print("  -> [PASS] NaN/Inf 없음 확인.")

    # ── 강화 8: failure 값 범위 검증 ──
    print("Validation 8: failure 컬럼 값 범위 {0,1} 검증")
    for name, path in [("TRAIN", output_train), ("VAL", output_val)]:
        fv = con.execute(f"SELECT DISTINCT failure FROM read_parquet('{path}')").fetchall()
        fset = set(r[0] for r in fv)
        assert fset.issubset({0, 1}), f"오류: {name} failure에 0/1 이외 값: {fset}"
    print("  -> [PASS] failure 값 범위 정상.")

    print("\n✅ [5-A단계 통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (8/8 PASS)")
finally:
    con.close()


🔍 [5-A단계 무결성 검증] 시작...
Validation 1: 출력 파일 존재 확인 검증
  -> [PASS] 출력 파일 존재 확인.
Validation 2: Train과 Validation 간 개체(serial_number) 중복 침수 검증
  -> [PASS] Train/Validation 간 개체 침수 없음.
Validation 3: 고장 개체와 정상 개체의 1배수(1:1) 샘플링 비율 검증
  -> [PASS] 고장 개체 대비 정상 개체 1배수 샘플링 통합성 확인.
Validation 4: 필수 컬럼 결측치 존재 여부 검증
  -> [PASS] 필수 메타 컬럼 결측치 없음.
Validation 5: 각 파일 row 수 > 0 검증
  -> TRAIN: 5,158,738 rows
  -> VAL: 1,251,905 rows
  -> [PASS] 모든 파일에 데이터 존재.
Validation 6: Train/Validation 스키마 일치 검증
  -> [PASS] Train/Validation 스키마 완전 일치.
Validation 7: NaN/Inf 잔존 검증
  -> [PASS] NaN/Inf 없음 확인.
Validation 8: failure 컬럼 값 범위 {0,1} 검증
  -> [PASS] failure 값 범위 정상.

✅ [5-A단계 통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (8/8 PASS)
